# Multi-Layer-EMA Strategy Research Notebook

## Objectif
Ameliorer le Sharpe ratio de 0.872 vers >1.0 en analysant:
1. Les periodes EMA optimales (fast/slow)
2. Les seuils de RSI et Bollinger Bands
3. Les parametres de stop-loss/trailing-stop
4. L'allocation entre BTC/ETH/LTC

## Performance actuelle
- Sharpe: 0.872
- CAGR: 40.4%
- Max DD: 50.5%
- Win Rate: 43%

In [ ]:
# Imports et setup QuantBook
from AlgorithmImports import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product

qb = QuantBook()

# Multi-crypto
btc = qb.add_crypto("BTCUSD", Resolution.HOUR).symbol
eth = qb.add_crypto("ETHUSD", Resolution.HOUR).symbol
ltc = qb.add_crypto("LTCUSD", Resolution.HOUR).symbol

symbols = [btc, eth, ltc]
print(f"Symbols: {[s.value for s in symbols]}")

Chargement des données historiques de prix et volumes pour l'analyse.

In [ ]:
# Chargement donnees etendues (2020-2025)
start_date = datetime(2020, 1, 1)
end_date = datetime(2025, 12, 31)

history = qb.history(symbols, start_date, end_date, Resolution.HOUR)
print(f"Data loaded: {len(history)} hourly bars")

# Reshape pour chaque symbole
dfs = {}
for symbol in symbols:
    df = history.loc[symbol].copy() if symbol in history.index.get_level_values(0) else None
    if df is not None:
        dfs[symbol.value] = df
        print(f"{symbol.value}: {len(df)} bars")

Calcul et analyse des métriques de performance du backtest.

In [ ]:
# Detection regimes de marche sur BTC
btc_df = dfs['BTCUSD'].copy()
btc_df['returns'] = btc_df['close'].pct_change()
btc_df['volatility_24h'] = btc_df['returns'].rolling(24).std() * np.sqrt(365 * 24)
btc_df['sma_200'] = btc_df['close'].rolling(200).mean()

# Resample to daily for regime detection
btc_daily = btc_df.resample('1D').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum'
})
btc_daily['returns'] = btc_daily['close'].pct_change()
btc_daily['volatility_daily'] = btc_daily['returns'].rolling(20).std() * np.sqrt(365)
btc_daily['sma_200'] = btc_daily['close'].rolling(200).mean()

# Visualisation
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

axes[0].semilogy(btc_daily.index, btc_daily['close'], label='BTC', alpha=0.8)
axes[0].semilogy(btc_daily.index, btc_daily['sma_200'], label='SMA200', linestyle='--', alpha=0.7)
axes[0].set_title('BTC Price & SMA200')
axes[0].legend()

axes[1].plot(btc_daily.index, btc_daily['volatility_daily'], label='Volatility (annualized)', color='orange')
axes[1].axhline(y=0.60, color='r', linestyle='--', label='60% threshold')
axes[1].set_title('BTC Daily Volatility')
axes[1].legend()

axes[2].bar(btc_daily.index, btc_daily['volume'] / 1e9, width=1, alpha=0.7)
axes[2].set_title('BTC Daily Volume (Billions)')

plt.tight_layout()
plt.show()

print(f"\nVolatility stats:")
print(btc_daily['volatility_daily'].describe())

Calcul des indicateurs techniques (moyennes mobiles, RSI, MACD, etc.) pour générer les signaux.

In [ ]:
# Grid search: EMA periods optimaux
fast_periods = [5, 8, 10, 12, 15, 20]
slow_periods = [30, 40, 50, 60, 80, 100]

results_ema = []

for fast in fast_periods:
    for slow in slow_periods:
        if fast >= slow:
            continue
        
        # Calcul EMA cross signals
        btc_daily['ema_fast'] = btc_daily['close'].ewm(span=fast).mean()
        btc_daily['ema_slow'] = btc_daily['close'].ewm(span=slow).mean()
        btc_daily['signal'] = (btc_daily['ema_fast'] > btc_daily['ema_slow']).astype(int)
        
        # Calcul returns
        btc_daily['strategy_returns'] = btc_daily['signal'].shift(1) * btc_daily['returns']
        btc_daily = btc_daily.dropna()
        
        if len(btc_daily) < 100:
            continue
        
        # Sharpe
        sharpe = btc_daily['strategy_returns'].mean() / btc_daily['strategy_returns'].std() * np.sqrt(365)
        total_return = (1 + btc_daily['strategy_returns']).prod() - 1
        max_dd = (btc_daily['strategy_returns'].cumsum() - btc_daily['strategy_returns'].cumsum().cummax()).min()
        trades = (btc_daily['signal'].diff() != 0).sum()
        
        results_ema.append({
            'fast': fast,
            'slow': slow,
            'sharpe': sharpe,
            'total_return': total_return,
            'max_dd': max_dd,
            'trades': trades
        })

results_ema_df = pd.DataFrame(results_ema)
results_ema_pivot = results_ema_df.pivot(index='fast', columns='slow', values='sharpe')

plt.figure(figsize=(10, 6))
plt.imshow(results_ema_pivot.values, cmap='RdYlGn', aspect='auto')
plt.colorbar(label='Sharpe Ratio')
plt.xticks(range(len(slow_periods)), slow_periods)
plt.yticks(range(len(fast_periods)), fast_periods)
plt.xlabel('Slow EMA Period')
plt.ylabel('Fast EMA Period')
plt.title('Sharpe Ratio: EMA Periods Grid Search')
plt.tight_layout()
plt.show()

best_ema = results_ema_df.loc[results_ema_df['sharpe'].idxmax()]
print(f"\nBest EMA periods: Fast={int(best_ema['fast'])}, Slow={int(best_ema['slow'])}")
print(f"Sharpe: {best_ema['sharpe']:.3f}, Return: {best_ema['total_return']:.1%}")

Construction des features et préparation des données pour l'apprentissage automatique.

In [ ]:
# Analyse correlation BTC/ETH/LTC pour allocation
print("\n=== CORRELATION ANALYSIS ===")

# Resample toutes les donnees en daily
daily_close = pd.DataFrame()
for symbol_name, df in dfs.items():
    daily = df.resample('1D').agg({'close': 'last'})
    daily_close[symbol_name] = daily['close']

# Returns
returns = daily_close.pct_change().dropna()

# Correlation matrix
corr_matrix = returns.corr()
print("\nCorrelation Matrix:")
print(corr_matrix)

# Stats par crypto
print("\nPerformance par crypto:")
for col in returns.columns:
    ann_return = returns[col].mean() * 365
    ann_vol = returns[col].std() * np.sqrt(365)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0
    print(f"{col}: Return={ann_return:.1%}, Vol={ann_vol:.1%}, Sharpe={sharpe:.2f}")

## Synthese et Recommandations

### Parametres optimaux identifies

In [ ]:
# Resume des recommandations
print("\n" + "="*60)
print("RECOMMANDATIONS POUR MULTI-LAYER-EMA")
print("="*60)

print(f"""
Parametres actuels:
- fastPeriod = 10
- slowPeriod = 50
- RSI range = 35-70
- fixed_stop_pct = 0.85
- trailing_stop_pct = 0.90
- take_profit_pct = 1.30
- volatility_threshold = 0.60

Parametres recommandes:
- fastPeriod = {int(best_ema['fast'])}
- slowPeriod = {int(best_ema['slow'])}
""")

import json
recommendations = {
    'fast_period': int(best_ema['fast']),
    'slow_period': int(best_ema['slow']),
}
print("\nJSON pour mise a jour du code:")
print(json.dumps(recommendations, indent=2))